In [928]:
# ToutAtrac :
# Zygarde50_Forme = % a gérer dans nettoyage + 681 shield et blade

# pd.sum(skipna =True)

### IMPORTS

In [929]:
import pandas as pd
import numpy as np
import csv

##### Chemin du fichier si local et dans dossier courant 

In [930]:
import os
chemin_dossier = os.getcwd()
nom_csv = "\\Pokemon_dataset.csv"

chemin_fichier = chemin_dossier + nom_csv
# recupère le dossier courant si csv dans meme dossier
# print(chemin_fichier)

##### Appel classe

In [931]:
import sys
# abspath transforme chemin relatif (construit avec join) en chemin absolu et os.padir = constante du module os qui contient la chaîne ".." = dossier parent
racine_projet = os.path.abspath(os.path.join(chemin_dossier, os.pardir))

if racine_projet not in sys.path:
    # sys.path = liste de chemins que Python utilise pour rechercher les modules importés
    # Python cherchera d’abord dans ce dossier
    sys.path.insert(0, racine_projet)

from classes.cleanup import PandasImport
from classes.uniformisation import PokemonDf

### Lecture csv et stockage DataFrame

Possible de transposer (.T) et changer axe colonne/ligne dans le pd.read csv

In [ ]:
def lire_csv_bon_encodage(chemin):

    def detect_sep(chemin):
        separateurs = ";,|\t"
        with open(chemin, newline='', encoding='utf-8', errors='ignore') as f:
            sample = f.read(2049) # échantillon des 2049 premiers caractères sur lequel trouver le séparateur
        dialect = csv.Sniffer().sniff(sample, delimiters=separateurs)
        return dialect.delimiter

    separateur = detect_sep(chemin)

    encodings = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]
    for enc in encodings:
        try:
            df = pd.read_csv(chemin, encoding=enc, sep=separateur)
            print(f"Lecture réussie avec l'encodage : {enc}")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"Erreur avec {enc} : {e}")
            continue

    raise ValueError("Aucun encodage n'a fonctionné pour ce fichier.")

df = lire_csv_bon_encodage(chemin_fichier)

print(df.head())
# print(f"\n{"_"*75}\n")
# print(df.describe(include="all"))
# print(f"\n{"_"*75}\n")
# print(df.info())
# print(f"\n{"_"*75}\n")

Lecture réussie avec l'encodage : utf-8
  Unnamed: 0          0         1          2                3          4  \
0          #        635       606         58              479         61   
1       Name  hydreigon  Beheeyem  Growlithe  RotomWash Rotom  Poliwhirl   
2      Total        600       485        350              520        385   
3         HP       92.0      75.0       55.0             50.0       65.0   
4     Attack        105        75         70               65         65   

         5       6          7       8  ...       790      791       792  \
0      560     317        442     690  ...       419      113       554   
1  SCRAFTY  Swalot  Spiritomb  skrelp  ...  FLOATZEL  CHANSEY  darumaka   
2      488     467        485     320  ...       495      450       315   
3      NaN   100.0       50.0    50.0  ...      85.0    250.0      70.0   
4       90      73         92      60  ...       105        5        90   

      793        794                    795     796 

### Premières manips

#####  Transpo, Infos

In [933]:
df_pokemon = df.T

In [934]:
print(df_pokemon.shape)
print(f"\n{"_"*75}\n")
print(df_pokemon.describe(include="all"))
print(f"\n{"_"*75}\n")
print(df.info())
print(f"\n{"_"*75}\n")
display(df_pokemon.head(3))
print(f"\n{"_"*75}\n")

(801, 11)

___________________________________________________________________________

         0     1    2     3    4      5        6     7    8      9         10
count   801   801  801   774  776    801      801   758  762    801       801
unique  722   801  201   103  111    400      500   109    8      3       155
top     479  Name  600  50.0  100  100.0  50 / 50  60.0  5.0  False  Normal, 
freq      6     1   37    62   39     69       12    42  158    735        61

___________________________________________________________________________

<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Columns: 801 entries, Unnamed: 0 to 799
dtypes: str(801)
memory usage: 69.0 KB
None

___________________________________________________________________________



,0,1,2,3,4,5,6,7,8,9,10
Unnamed: 0,#,Name,Total,HP,Attack,Defense % of Attack,Sp. Atk / Sp. Def,Speed,Generation,Legendary,Types
0,635,hydreigon,600,92.0,105,85.7143,125 / 90,98.0,5.0,False,"Dark, Dragon"
1,606,Beheeyem,485,75.0,75,100.0,125 / 95,40.0,5.0,False,"Psychic,"



___________________________________________________________________________



##### Copie DataFrame pour pouvoir l'appeler plus loin pour des vérificatons

In [935]:
df_pokemon_base = df_pokemon.copy()

### Utilitaires

##### Fonction pour voir toutes les lignes

In [936]:
def see_all(df :pd.DataFrame | pd.Series, limit :int | None = None, shape_on = False):
    with pd.option_context("display.max_rows", limit):
        display(df)
        if shape_on == True:
            print(df.shape)

##### Fonction séparation de colonne

In [937]:
# séparation sur séparateur spécifique underscore (_)
def split_column(df, col_name, new_cols):
    ind_col = df.columns.get_loc(col_name)

    split_df = df[col_name].str.split("_", n=2, expand=True)[[0, 1]]
    # [[0, 1]] liste qui permet de ne garder que les deux premiers splits pour cas particulier 3 valeurs dans colonne de base et les deux derniers étaient des doublons 
    split_df.columns = new_cols

    if len(new_cols) > 1:
        split_df[new_cols[1]] = split_df[new_cols[1]].fillna("")

    df = df.drop(columns=[col_name])

    df = pd.concat(
        [df.iloc[:, :ind_col], split_df, df.iloc[:, ind_col:]],
        axis=1
    )
    return df

##### Masques

In [938]:
def choose_mask(df, cols=None, mask=None):
    """
    Retourne un masque 
    """
    if mask == "negativ":
        if cols is None:
            raise ValueError("Il faut donner une `cols`'")
        return (df[cols] < 0).any(axis=1)
  
    if mask == "one_nan":
        if cols is None:
            raise ValueError("Il faut donner une `cols`'")
        return (df[cols].isna().sum(axis=1)) == 1
    
    if mask == "valeur_trop_grande":
        if cols is None:
            raise ValueError("Il faut donner une `cols`'")
        return (df[cols].abs() >= 1000).any(axis=1)
        
    raise ValueError(f"Masque inconnu : {mask}")

##### Fonction voir datas manquantes ou erronées

In [939]:
def get_missing(df :pd.DataFrame, col :str) -> pd.DataFrame:
    """
    Retourne subset (sous dataframe) avec valeur manquante sur colonnes voulues
    """
    return df[df[col].isna()]

def see_missing(df :pd.DataFrame, col :str, limit :int | None = None) -> None:
    """
    col = total pour voir le total des données manquantes par colonne 
    col = sum_total pour voir le nombre de données manquantes dans tout le df
    sinon choisir la colonne voulue et ensuite la limite de lignes affichées
    + paramètre de see_all a True pour voir le shape du df retourné
    """
    if col == "total":
        subset = df.isna().sum()
    elif col == "sum_total":
        subset = df.isna().sum().sum()
    else:
        subset = get_missing(df, col)

    if isinstance(limit, int) :
        see_all(subset, limit)
    else:
        see_all(subset, limit, True)

In [940]:
def see_mask(df, mask, limit=None, shape_on=False):
    """
    Affiche uniquement les lignes retenues par un masque.
    """
    subset = df.loc[mask].copy()
    see_all(subset, limit=limit, shape_on=shape_on)
    return 

def get_mask(df, mask):
    """
    Affiche uniquement les lignes retenues par un masque.
    """
    subset = df.loc[mask].copy()
    return subset

In [941]:
def compare_col_sum_cols(df, col_total, cols_to_sum):
    if col_total not in df.columns:
        raise KeyError(f"Colonne inconnue : {col_total}")

    for col in cols_to_sum:
        if col not in df.columns:
            raise KeyError(f"Colonnes manquantes pour le calcul : {col}")

    expected_total = df[col_total]
    sum_col = df[cols_to_sum].sum(axis=1)

    mask_ok = expected_total.eq(sum_col)
    mask_no = ~mask_ok
    
    invalid_rows = df.loc[mask_no, cols_to_sum + [col_total]].copy()
    invalid_rows["total_attendu"] = sum_col.loc[mask_no]
    invalid_rows["difference"] = invalid_rows[col_total] - invalid_rows["total_attendu"]
    
    # if mask_no.any():
    #     display(df.loc[mask_no])
    return invalid_rows

##### Fonctions calculs pour combler datas manquantes

In [942]:
def calcul_pourcentage_ratio(df, col_drop, col1, col2): 
    if col_drop not in df.columns:
        return df
    else:
        ind_col = df.columns.get_loc(col_drop)
        new_col = (df[col1] * (df[col_drop] / 100)).round(1)
        df = df.drop(columns=[col_drop])
        df.insert(ind_col, col2, new_col)
    return df 

In [943]:
def set_mask(df, mask, cols_to_update, transform):
    """
    Applique une transformation sur les lignes sélectionnées par un masque.
    """
    if isinstance(cols_to_update, str):
        cols_to_update = [cols_to_update]

    if callable(transform):
        df.loc[mask, cols_to_update] = transform(df.loc[mask, cols_to_update])
    else:
        df.loc[mask, cols_to_update] = transform

    return df

#### Déclaration de variables

##### Variables des fonctions de la classe cleanup

In [944]:
do_lower = True
keep_lower = ["IMBD"]
do_capitalize = True
keep_capitalize = ["IMBD"]
del_punctuation = True
keep_punctuation = [":", "_", "°"] # moyen a trouver pour 50% pokemon reste en exception 
del_accent = True
spaces_to_sep = True
sep = '_'

##### Variables contenant labels des colonnes

In [945]:
col_total = ["total"]
cols_sum_in_total =  ["hp", "attack", "defense", "sp_atck", "sp_def", "speed"]
all_cols = col_total + cols_sum_in_total + ["n°", "name", "generation", "legendary", "type_1", "type_2"]

### Nettoyage

#### HEADER

##### Clean index & renommer labels colonnes

In [946]:
df_pokemon = df.T # pour reset si je relance la cellule, sinon colonne suivante écrase la précédente

# crée et ajoute une colonne à gauche de la table avec des valeurs allant de 0 à la taille de la dataframe avec un pas de 1 = permet de recréer un index propre (0, 1, 2, ...), ce qui évite un index corrompu ou non aligné. (si pas de données vitales dasn l'index) 
df_pokemon = df_pokemon.reset_index(drop=True)
# argument drop=True: supprime la colonne la plus à gauche avant l'ajout des nouveaux indices

# écrase valeur labels avec les valeurs présentes dans la [ligne donnée]
df_pokemon.columns = df_pokemon.iloc[0]
# supprimer le doublon après copie
df_pokemon = df_pokemon.iloc[1:].reset_index(drop=True)

df_pokemon = df_pokemon.rename(columns={"#": "N°"})

display(df_pokemon)

,N°,Name,Total,HP,Attack,Defense % of Attack,Sp. Atk / Sp. Def,Speed,Generation,Legendary,Types
0,635,hydreigon,600,92.0,105,85.7143,125 / 90,98.0,5.0,False,"Dark, Dragon"
1,606,Beheeyem,485,75.0,75,100.0,125 / 95,40.0,5.0,False,"Psychic,"
2,58,Growlithe,350,55.0,70,64.2857,70 / 50,60.0,1.0,False,"Fire,"
3,479,RotomWash Rotom,520,50.0,65,164.6154,105 / 107,86.0,4.0,False,"Electric, Water"
4,61,Poliwhirl,385,65.0,65,100.0,50 / 50,90.0,1.0,False,"Water,"
...,...,...,...,...,...,...,...,...,...,...,...
795,65,ALAKAZAMMEGA ALAKAZAM,590,55.0,NaN,130.0,175 / 95,150.0,1.0,False,"Psychic,"
796,98,KRABBY,325,30.0,105,85.7143,25 / 25,50.0,1.0,False,"Water,"
797,250,Ho-oh,680,106.0,130,69.2308,110 / 154,90.0,2.0,True,"Fire, Flying"
798,390,chimchar,309,NaN,58,75.8621,58 / 44,61.0,NaN,False,"Fire,"


##### Cleanup

In [947]:
# Initial list of headers:
*cols, = df_pokemon
print("\nInitial name :\n", cols, "\n")

# Cleaning headers with PandasImport.final_clean (fonction de la classe importée):
pd_tool_clean = PandasImport()

new_label_cols = [
    pd_tool_clean.final_clean(
        str(col_header),
        do_lower=do_lower,
        keep_lower=keep_lower,
        del_punctuation=del_punctuation,
        keep_punctuation=keep_punctuation,
        del_accent=del_accent,
        spaces_to_sep=spaces_to_sep
    )
    for col_header in cols
]
print("\nCleaned columns:\n", new_label_cols)

mapping_header = {old_name: new_name for old_name, new_name in zip(cols, new_label_cols)}
df_pokemon = df_pokemon.rename(columns=mapping_header)
print("\nFinal DataFrame")
display(df_pokemon)


Initial name :
 ['N°', 'Name', 'Total', 'HP', 'Attack', 'Defense % of Attack', 'Sp. Atk / Sp. Def', 'Speed', 'Generation', 'Legendary', 'Types'] 


Cleaned columns:
 ['n°', 'name', 'total', 'hp', 'attack', 'defense_of_attack', 'sp_atk_sp_def', 'speed', 'generation', 'legendary', 'types']

Final DataFrame


,n°,name,total,hp,attack,defense_of_attack,sp_atk_sp_def,speed,generation,legendary,types
0,635,hydreigon,600,92.0,105,85.7143,125 / 90,98.0,5.0,False,"Dark, Dragon"
1,606,Beheeyem,485,75.0,75,100.0,125 / 95,40.0,5.0,False,"Psychic,"
2,58,Growlithe,350,55.0,70,64.2857,70 / 50,60.0,1.0,False,"Fire,"
3,479,RotomWash Rotom,520,50.0,65,164.6154,105 / 107,86.0,4.0,False,"Electric, Water"
4,61,Poliwhirl,385,65.0,65,100.0,50 / 50,90.0,1.0,False,"Water,"
...,...,...,...,...,...,...,...,...,...,...,...
795,65,ALAKAZAMMEGA ALAKAZAM,590,55.0,NaN,130.0,175 / 95,150.0,1.0,False,"Psychic,"
796,98,KRABBY,325,30.0,105,85.7143,25 / 25,50.0,1.0,False,"Water,"
797,250,Ho-oh,680,106.0,130,69.2308,110 / 154,90.0,2.0,True,"Fire, Flying"
798,390,chimchar,309,NaN,58,75.8621,58 / 44,61.0,NaN,False,"Fire,"


#### DATAFRAME : cleanup

##### clean cellules

In [948]:
def clean_cell(value):
    if isinstance(value, str):
        return pd_tool_clean.final_clean(
        # fonction classe cleanup
            value,
            do_lower=do_lower,
            keep_lower=keep_lower,
            del_punctuation=del_punctuation,
            keep_punctuation=keep_punctuation,
            del_accent=del_accent,
            spaces_to_sep=spaces_to_sep,
            sep=sep
        )
    return value

df_pokemon[["name", "types", "sp_atk_sp_def"]] = df_pokemon[["name", "types", "sp_atk_sp_def"]].map(clean_cell)

see_all(df_pokemon, 10)

,n°,name,total,hp,attack,defense_of_attack,sp_atk_sp_def,speed,generation,legendary,types
0,635,hydreigon,600,92.0,105,85.7143,125_90,98.0,5.0,False,dark_dragon
1,606,beheeyem,485,75.0,75,100.0,125_95,40.0,5.0,False,psychic
2,58,growlithe,350,55.0,70,64.2857,70_50,60.0,1.0,False,fire
3,479,rotomwash_rotom,520,50.0,65,164.6154,105_107,86.0,4.0,False,electric_water
4,61,poliwhirl,385,65.0,65,100.0,50_50,90.0,1.0,False,water
...,...,...,...,...,...,...,...,...,...,...,...
795,65,alakazammega_alakazam,590,55.0,NaN,130.0,175_95,150.0,1.0,False,psychic
796,98,krabby,325,30.0,105,85.7143,25_25,50.0,1.0,False,water
797,250,hooh,680,106.0,130,69.2308,110_154,90.0,2.0,True,fire_flying
798,390,chimchar,309,NaN,58,75.8621,58_44,61.0,NaN,False,fire


##### Si jamais besoin de gérer les doublons

In [949]:
# df_pokemon = df_pokemon.drop_duplicates()

##### spliter attaque et défense spéciales

In [950]:
df_pokemon = split_column(df_pokemon, "sp_atk_sp_def", ["sp_atck", "sp_def"])
display(df_pokemon)

,n°,name,total,hp,attack,defense_of_attack,sp_atck,sp_def,speed,generation,legendary,types
0,635,hydreigon,600,92.0,105,85.7143,125,90,98.0,5.0,False,dark_dragon
1,606,beheeyem,485,75.0,75,100.0,125,95,40.0,5.0,False,psychic
2,58,growlithe,350,55.0,70,64.2857,70,50,60.0,1.0,False,fire
3,479,rotomwash_rotom,520,50.0,65,164.6154,105,107,86.0,4.0,False,electric_water
4,61,poliwhirl,385,65.0,65,100.0,50,50,90.0,1.0,False,water
...,...,...,...,...,...,...,...,...,...,...,...,...
795,65,alakazammega_alakazam,590,55.0,NaN,130.0,175,95,150.0,1.0,False,psychic
796,98,krabby,325,30.0,105,85.7143,25,25,50.0,1.0,False,water
797,250,hooh,680,106.0,130,69.2308,110,154,90.0,2.0,True,fire_flying
798,390,chimchar,309,NaN,58,75.8621,58,44,61.0,NaN,False,fire


##### format noms et cellules float

In [951]:
pd_tools = PokemonDf()

df_pokemon = pd_tools.format_columns_names(df_pokemon, ["name", "types"])
display(df_pokemon)

,n°,name,total,hp,attack,defense_of_attack,sp_atck,sp_def,speed,generation,legendary,types
0,635,Hydreigon,600,92.0,105,85.7143,125,90,98.0,5.0,False,Dark_Dragon
1,606,Beheeyem,485,75.0,75,100.0,125,95,40.0,5.0,False,Psychic
2,58,Growlithe,350,55.0,70,64.2857,70,50,60.0,1.0,False,Fire
3,479,RotomWash,520,50.0,65,164.6154,105,107,86.0,4.0,False,Electric_Water
4,61,Poliwhirl,385,65.0,65,100.0,50,50,90.0,1.0,False,Water
...,...,...,...,...,...,...,...,...,...,...,...,...
795,65,AlakazamMega,590,55.0,NaN,130.0,175,95,150.0,1.0,False,Psychic
796,98,Krabby,325,30.0,105,85.7143,25,25,50.0,1.0,False,Water
797,250,Hooh,680,106.0,130,69.2308,110,154,90.0,2.0,True,Fire_Flying
798,390,Chimchar,309,NaN,58,75.8621,58,44,61.0,NaN,False,Fire


In [952]:
df_pokemon = pd_tools.format_columns_to_float(df_pokemon, [
"total", "hp", "attack", "defense_of_attack", "sp_atck", "sp_def", "speed", "generation"
])

display(df_pokemon)

,n°,name,total,hp,attack,defense_of_attack,sp_atck,sp_def,speed,generation,legendary,types
0,635,Hydreigon,600.0,92.0,105.0,85.7143,125.0,90.0,98.0,5.0,False,Dark_Dragon
1,606,Beheeyem,485.0,75.0,75.0,100.0000,125.0,95.0,40.0,5.0,False,Psychic
2,58,Growlithe,350.0,55.0,70.0,64.2857,70.0,50.0,60.0,1.0,False,Fire
3,479,RotomWash,520.0,50.0,65.0,164.6154,105.0,107.0,86.0,4.0,False,Electric_Water
4,61,Poliwhirl,385.0,65.0,65.0,100.0000,50.0,50.0,90.0,1.0,False,Water
...,...,...,...,...,...,...,...,...,...,...,...,...
795,65,AlakazamMega,590.0,55.0,NaN,130.0000,175.0,95.0,150.0,1.0,False,Psychic
796,98,Krabby,325.0,30.0,105.0,85.7143,25.0,25.0,50.0,1.0,False,Water
797,250,Hooh,680.0,106.0,130.0,69.2308,110.0,154.0,90.0,2.0,True,Fire_Flying
798,390,Chimchar,309.0,NaN,58.0,75.8621,58.0,44.0,61.0,NaN,False,Fire


##### masque pour vérifier exceptions toutes gérées

In [975]:
df_pokemon["name"] = df_pokemon["name"].str.title()

pd_tools = PokemonDf()

see_all(df_pokemon[pd_tools.mask_exc_name(df_pokemon)], shape_on=True)

,n°,name,total,hp,attack,defense,sp_atck,sp_def,speed,generation,legendary,type_1,type_2
144,6,Charizardmega_X,634.0,78.0,130.0,111.0,130.0,85.0,100.0,1.0,False,Fire,Dragon
559,6,Charizardmega_Y,634.0,78.0,104.0,78.0,159.0,115.0,100.0,1.0,False,Fire,Flying
204,122,Mr_Mime,460.0,40.0,45.0,65.0,100.0,120.0,90.0,1.0,False,Psychic,Fairy
214,150,Mewtwomega_Y,780.0,106.0,150.0,70.0,194.0,120.0,140.0,1.0,True,Psychic,
288,150,Mewtwomega_X,780.0,106.0,190.0,100.0,154.0,100.0,130.0,1.0,True,Psychic,Fighting
429,439,Mime_Jr,310.0,20.0,25.0,45.0,70.0,90.0,60.0,4.0,False,Psychic,Fairy
377,718,Zygarde50_Forme,600.0,108.0,100.0,121.0,81.0,95.0,95.0,6.0,True,Dragon,Ground
156,720,Hoopahoopa_Unbound,680.0,80.0,160.0,60.0,170.0,130.0,80.0,6.0,True,Psychic,Dark
718,720,Hoopahoopa_Confined,600.0,80.0,110.0,60.0,150.0,130.0,70.0,6.0,True,Psychic,Ghost


(9, 13)


### Tri

##### Trier par numéro pokédex

In [954]:
df_pokemon = df_pokemon.sort_values("n°", key=lambda numero_pokemon: pd.to_numeric(numero_pokemon, downcast="integer"))

# si beosin reset index une deuxième fois
# df_pokemon = df_pokemon.reset_index(drop=True)
display(df_pokemon)

,n°,name,total,hp,attack,defense_of_attack,sp_atck,sp_def,speed,generation,legendary,types
374,1,Bulbasaur,318.0,45.0,49.0,100.0000,65.0,65.0,45.0,NaN,False,Grass_Poison
729,2,Ivysaur,405.0,-60.0,62.0,101.6129,80.0,80.0,60.0,NaN,False,Grass_Poison
151,3,Venusaur,525.0,80.0,82.0,101.2195,100.0,100.0,80.0,1.0,False,Grass_Poison
325,3,Venusaurmega,625.0,80.0,100.0,123.0000,122.0,120.0,80.0,1.0,False,Grass_Poison
672,4,Charmander,309.0,39.0,52.0,82.6923,60.0,50.0,65.0,1.0,False,Fire
...,...,...,...,...,...,...,...,...,...,...,...,...
38,719,Dianciemega,700.0,50.0,160.0,68.7500,160.0,110.0,110.0,6.0,True,Rock_Fairy
72,719,Diancie,600.0,50.0,100.0,150.0000,100.0,150.0,50.0,6.0,True,Rock_Fairy
156,720,Hoopahoopa_Unbound,680.0,80.0,160.0,37.5000,170.0,130.0,80.0,6.0,True,Psychic_Dark
718,720,Hoopahoopa_Confined,600.0,80.0,110.0,54.5455,150.0,130.0,70.0,6.0,True,Psychic_Ghost


In [955]:
display(df_pokemon.sort_index(na_position="first"))

,n°,name,total,hp,attack,defense_of_attack,sp_atck,sp_def,speed,generation,legendary,types
0,635,Hydreigon,600.0,92.0,105.0,85.7143,125.0,90.0,98.0,5.0,False,Dark_Dragon
1,606,Beheeyem,485.0,75.0,75.0,100.0000,125.0,95.0,40.0,5.0,False,Psychic
2,58,Growlithe,350.0,55.0,70.0,64.2857,70.0,50.0,60.0,1.0,False,Fire
3,479,Rotomwash,520.0,50.0,65.0,164.6154,105.0,107.0,86.0,4.0,False,Electric_Water
4,61,Poliwhirl,385.0,65.0,65.0,100.0000,50.0,50.0,90.0,1.0,False,Water
...,...,...,...,...,...,...,...,...,...,...,...,...
795,65,Alakazammega,590.0,55.0,NaN,130.0000,175.0,95.0,150.0,1.0,False,Psychic
796,98,Krabby,325.0,30.0,105.0,85.7143,25.0,25.0,50.0,1.0,False,Water
797,250,Hooh,680.0,106.0,130.0,69.2308,110.0,154.0,90.0,2.0,True,Fire_Flying
798,390,Chimchar,309.0,NaN,58.0,75.8621,58.0,44.0,61.0,NaN,False,Fire


##### Spliter les types

In [956]:
df_pokemon = split_column(df_pokemon, "types", ["type_1", "type_2"])

display(df_pokemon)

,n°,name,total,hp,attack,defense_of_attack,sp_atck,sp_def,speed,generation,legendary,type_1,type_2
374,1,Bulbasaur,318.0,45.0,49.0,100.0000,65.0,65.0,45.0,NaN,False,Grass,Poison
729,2,Ivysaur,405.0,-60.0,62.0,101.6129,80.0,80.0,60.0,NaN,False,Grass,Poison
151,3,Venusaur,525.0,80.0,82.0,101.2195,100.0,100.0,80.0,1.0,False,Grass,Poison
325,3,Venusaurmega,625.0,80.0,100.0,123.0000,122.0,120.0,80.0,1.0,False,Grass,Poison
672,4,Charmander,309.0,39.0,52.0,82.6923,60.0,50.0,65.0,1.0,False,Fire,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
38,719,Dianciemega,700.0,50.0,160.0,68.7500,160.0,110.0,110.0,6.0,True,Rock,Fairy
72,719,Diancie,600.0,50.0,100.0,150.0000,100.0,150.0,50.0,6.0,True,Rock,Fairy
156,720,Hoopahoopa_Unbound,680.0,80.0,160.0,37.5000,170.0,130.0,80.0,6.0,True,Psychic,Dark
718,720,Hoopahoopa_Confined,600.0,80.0,110.0,54.5455,150.0,130.0,70.0,6.0,True,Psychic,Ghost


### Calculs sur datas

In [ ]:
# A VERIFIER : une erreur de plus sur df_pokemon par rapport au DataFrame de base
# display(df_pokemon_base[df_pokemon_base[8].isna()].shape)
# display(df_pokemon_base[df_pokemon_base[8].isna()])

see_missing(df_pokemon, col = "total")

n°                    0
name                  0
total                 0
hp                   27
attack               26
defense_of_attack     0
sp_atck               0
sp_def                0
speed                43
generation           40
legendary             0
type_1                0
type_2                0
dtype: int64

(13,)


##### Calculer la défense sur l'attaque (defense % attaque)

In [958]:
df_pokemon = calcul_pourcentage_ratio(df_pokemon, "defense_of_attack", "attack", "defense")

display(df_pokemon)

,n°,name,total,hp,attack,defense,sp_atck,sp_def,speed,generation,legendary,type_1,type_2
374,1,Bulbasaur,318.0,45.0,49.0,49.0,65.0,65.0,45.0,NaN,False,Grass,Poison
729,2,Ivysaur,405.0,-60.0,62.0,63.0,80.0,80.0,60.0,NaN,False,Grass,Poison
151,3,Venusaur,525.0,80.0,82.0,83.0,100.0,100.0,80.0,1.0,False,Grass,Poison
325,3,Venusaurmega,625.0,80.0,100.0,123.0,122.0,120.0,80.0,1.0,False,Grass,Poison
672,4,Charmander,309.0,39.0,52.0,43.0,60.0,50.0,65.0,1.0,False,Fire,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
38,719,Dianciemega,700.0,50.0,160.0,110.0,160.0,110.0,110.0,6.0,True,Rock,Fairy
72,719,Diancie,600.0,50.0,100.0,150.0,100.0,150.0,50.0,6.0,True,Rock,Fairy
156,720,Hoopahoopa_Unbound,680.0,80.0,160.0,60.0,170.0,130.0,80.0,6.0,True,Psychic,Dark
718,720,Hoopahoopa_Confined,600.0,80.0,110.0,60.0,150.0,130.0,70.0,6.0,True,Psychic,Ghost


##### Vérifier si total est bon et si données manquantes // 1 fois

In [959]:
df_compare_total = compare_col_sum_cols(df_pokemon, "total", cols_sum_in_total )

display(df_compare_total)

,hp,attack,defense,sp_atck,sp_def,speed,total,total_attendu,difference
729,-60.0,62.0,63.0,80.0,80.0,60.0,405.0,285.0,120.0
267,15925.0,90.0,40.0,45.0,80.0,75.0,395.0,16255.0,-15860.0
734,55.0,47.0,52.0,40.0,40.0,NaN,275.0,234.0,41.0
428,NaN,80.0,85.0,110.0,90.0,50.0,490.0,415.0,75.0
98,NaN,65.0,60.0,90.0,75.0,90.0,450.0,380.0,70.0
...,...,...,...,...,...,...,...,...,...
157,-75.0,80.0,60.0,65.0,90.0,102.0,472.0,322.0,150.0
540,60.0,50.0,150.0,50.0,150.0,NaN,520.0,460.0,60.0
536,50.0,50.0,150.0,50.0,150.0,NaN,500.0,450.0,50.0
611,NaN,75.0,53.0,83.0,113.0,60.0,452.0,384.0,68.0


##### Valeurs négatives

In [960]:
mask_neg = choose_mask(df_pokemon, cols_sum_in_total, mask="negativ")
# see_neg = see_mask(df_compare_total, mask_neg, shape_on=True)

# ici toutes les valeurs négatives si passent en positif corrige le total donc je les repasse simplement en positif
df_pokemon = set_mask(
    df_pokemon,
    mask_neg,
    cols_to_update=cols_sum_in_total,
    transform=lambda subset: subset.abs()
)

df_compare_total = compare_col_sum_cols(df_pokemon, "total", cols_sum_in_total)
display(df_compare_total)

see_mask(df_pokemon, mask_neg, shape_on=True)

,hp,attack,defense,sp_atck,sp_def,speed,total,total_attendu,difference
267,15925.0,90.0,40.0,45.0,80.0,75.0,395.0,16255.0,-15860.0
734,55.0,47.0,52.0,40.0,40.0,NaN,275.0,234.0,41.0
428,NaN,80.0,85.0,110.0,90.0,50.0,490.0,415.0,75.0
98,NaN,65.0,60.0,90.0,75.0,90.0,450.0,380.0,70.0
185,80.0,82.0,78.0,95.0,80.0,NaN,500.0,415.0,85.0
...,...,...,...,...,...,...,...,...,...
641,66.0,NaN,NaN,62.0,57.0,52.0,350.0,237.0,113.0
540,60.0,50.0,150.0,50.0,150.0,NaN,520.0,460.0,60.0
536,50.0,50.0,150.0,50.0,150.0,NaN,500.0,450.0,50.0
611,NaN,75.0,53.0,83.0,113.0,60.0,452.0,384.0,68.0


,n°,name,total,hp,attack,defense,sp_atck,sp_def,speed,generation,legendary,type_1,type_2
729,2,Ivysaur,405.0,60.0,62.0,63.0,80.0,80.0,60.0,NaN,False,Grass,Poison
665,494,Victini,600.0,100.0,100.0,100.0,100.0,100.0,100.0,5.0,True,Psychic,Fire
661,504,Patrat,255.0,45.0,55.0,39.0,35.0,39.0,42.0,5.0,False,Normal,
560,590,Foongus,294.0,69.0,55.0,45.0,55.0,55.0,15.0,5.0,False,Grass,Poison
157,676,Furfrou,472.0,75.0,80.0,60.0,65.0,90.0,102.0,6.0,False,Normal,


(5, 13)


##### Valeurs trop grandes

In [961]:
mask_val_sup = choose_mask(df_pokemon, cols_sum_in_total, mask="valeur_trop_grande")
# see_mask(df_pokemon, mask=mask_val_sup)

df_pokemon = set_mask(
    df_pokemon,
    mask_val_sup,
    cols_to_update="hp",
    transform=np.nan
)

# beosin de réecraser mon masque pour voir les valeurs après focntion
mask_val_sup_check = choose_mask(df_pokemon, cols_sum_in_total, mask="valeur_trop_grande")
see_mask(df_pokemon, mask_val_sup_check, shape_on=True)

,n°,name,total,hp,attack,defense,sp_atck,sp_def,speed,generation,legendary,type_1,type_2


(0, 13)


##### Générations manquantes

In [962]:
# see_all(get_missing(df_pokemon, "generation"), shape_on=True) # = 40

sub_gen_nan = df_pokemon["generation"].isna() # pour avoir un deuxieme bool a comparer avec autre masque, pas un subset comme avec ma fonction 
foward_val = df_pokemon["generation"].ffill()
before_val = df_pokemon["generation"].bfill()

# ------ notna() = bool not missing ------
mask_method_ambiguous = sub_gen_nan & foward_val.notna() & before_val.notna() & (foward_val != before_val)
mask_method_ok = sub_gen_nan & ~mask_method_ambiguous

# see_mask(df_pokemon, mask_method_ok)
# see_mask(df_pokemon, mask_method_ambiguous)

df_pokemon = set_mask(
    df_pokemon,
    mask_method_ok,
    cols_to_update = "generation",
    transform = lambda subset: foward_val.loc[subset.index].fillna(before_val.loc[subset.index])
)

see_missing(df_pokemon, "generation")

,n°,name,total,hp,attack,defense,sp_atck,sp_def,speed,generation,legendary,type_1,type_2
213,151,Mew,600.0,100.0,100.0,100.0,100.0,100.0,100.0,NaN,False,Psychic,
749,152,Chikorita,318.0,45.0,49.0,65.0,49.0,65.0,45.0,NaN,False,Grass,
101,387,Turtwig,318.0,55.0,68.0,64.0,45.0,55.0,31.0,NaN,False,Grass,
308,388,Grotle,405.0,75.0,89.0,85.0,55.0,65.0,36.0,NaN,False,Grass,
477,493,Arceus,720.0,120.0,120.0,120.0,120.0,120.0,120.0,NaN,True,Normal,


(5, 13)


In [963]:
# pour les 5 valeurs ambigües je traite a la main avec une recherche des vraies générations, même pour celles ou j'aurais pu relancer mon bloc précédetn (évite de refaire un deuxième passage)

corrections = {213: 1, 749: 2, 101: 4, 308: 4, 477: 4}  

for idx, gen in corrections.items():
    df_pokemon.loc[idx, "generation"] = gen


see_missing(df_pokemon, "generation")

,n°,name,total,hp,attack,defense,sp_atck,sp_def,speed,generation,legendary,type_1,type_2


(0, 13)


##### Une valeur manquante (un NaN)

In [964]:
sub_diff_sur_total = compare_col_sum_cols(df_pokemon, "total", cols_sum_in_total)

mask_one_nan = choose_mask(df_pokemon, cols_sum_in_total, mask="one_nan")
sub_one_nan_total_diff = get_mask(sub_diff_sur_total, mask_one_nan)

valeurs_diff = sub_one_nan_total_diff["difference"]

# dans ma fonction transform(df.loc[mask, cols_to_update]), donc ici subset = df.loc[mask, cols_to_update]
transform_nan_par_diff = lambda subset: subset.apply(
        lambda col: col.where(~col.isna(), valeurs_diff))

df_pokemon = set_mask(
        df_pokemon,
        mask_one_nan,
        cols_to_update = cols_sum_in_total,
        transform = transform_nan_par_diff
)

In [965]:
see_missing(df_pokemon, col = "total")

n°             0
name           0
total          0
hp             5
attack        26
defense       26
sp_atck        0
sp_def         0
speed          4
generation     0
legendary      0
type_1         0
type_2         0
dtype: int64

(13,)


##### Deux NaN / pas fait

In [967]:
check_nan_rest = compare_col_sum_cols(df_pokemon, "total", cols_sum_in_total)
see_all (check_nan_rest, shape_on=True)

,hp,attack,defense,sp_atck,sp_def,speed,total,total_attendu,difference
362,25.0,NaN,NaN,105.0,55.0,90.0,310.0,275.0,35.0
795,55.0,NaN,NaN,175.0,95.0,150.0,590.0,475.0,115.0
358,50.0,NaN,NaN,70.0,30.0,40.0,300.0,190.0,110.0
799,60.0,NaN,NaN,170.0,95.0,130.0,600.0,455.0,145.0
232,130.0,NaN,NaN,110.0,95.0,65.0,525.0,400.0,125.0
772,61.0,NaN,NaN,70.0,70.0,70.0,420.0,271.0,149.0
197,NaN,60.0,40.0,40.0,40.0,NaN,250.0,180.0,70.0
131,40.0,NaN,NaN,30.0,30.0,30.0,220.0,130.0,90.0
742,70.0,NaN,NaN,43.0,53.0,40.0,302.0,206.0,96.0
167,130.0,NaN,NaN,70.0,35.0,60.0,400.0,295.0,105.0


(30, 9)


In [980]:
display(df_pokemon[df_pokemon["n°"] == "65"])

,n°,name,total,hp,attack,defense,sp_atck,sp_def,speed,generation,legendary,type_1,type_2
795,65,Alakazammega,590.0,55.0,NaN,NaN,175.0,95.0,150.0,1.0,False,Psychic,
206,65,Alakazam,500.0,55.0,50.0,45.0,135.0,95.0,120.0,1.0,False,Psychic,
